In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
from estival.sampling import tools as esamp
from tbdynamics.calib_utils import plot_spaghetti, plot_output_ranges, get_bcm
from tbdynamics.inputs import load_targets
import arviz as az
from typing import List, Dict
from numpyro import distributions as dist
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
OUT_PATH = Path.cwd() / 'runs/r2504'

In [ ]:
idata = az.from_netcdf(OUT_PATH / 'calib_full_out.nc')
idata = idata.sel(chain= [0,1,2,3], draw=slice(25000,None))
idata_extract = az.extract(idata, num_samples=500)

In [ ]:
az.plot_trace(idata, figsize=(16,3.2*len(idata.posterior)),compact=False)

In [ ]:
az.summary(idata)

In [ ]:
params = {
    # "contact_rate": 0.007255277552812773,
    "start_population_size": 2200000,
    # "rr_infection_latent": 0.4435087078206438,
    # "rr_infection_recovered": 0.3726339126072788,
    # "progression_multiplier": 1.71734470040371,
    "seed_time": 1810.0,
    "seed_num": 70.0,
    "seed_duration": 10.0,
    # "smear_positive_death_rate": 0.3789083132417181,
    # "smear_negative_death_rate": 0.023640460692223537,
    # "smear_positive_self_recovery": 0.2064856493052769,
    # "smear_negative_self_recovery": 0.11108908197816095,
    # "screening_scaleup_shape": 0.1289863504422179,
    #"screening_inflection_time": 2000,
    # "screening_end_asymp": 0.5820504987396568,
    # "detection_reduction": 0.7542772690107872,
}
bcm = get_bcm(params)

In [ ]:
spaghetti = esamp.model_results_for_samples(idata_extract, bcm)

In [ ]:
quantiles = [0.025, 0.25, 0.5, 0.75, 0.975]
quantile_outputs = esamp.quantiles_for_results(spaghetti, quantiles)
targets = load_targets()

In [ ]:
plot_spaghetti(spaghetti, ['total_population','notification'], 2, targets)

In [ ]:
plot_spaghetti(spaghetti, ['prevalence_pulmonary','incidence'], 2, targets)

In [ ]:
plot_output_ranges(quantile_outputs,targets, ['total_population','notification'], quantiles, 1, 1950, 2025)

In [ ]:
plot_output_ranges(quantile_outputs,targets, ['prevalence_pulmonary','incidence', 'percentage_latent'], quantiles, 1, 2000, 2025)

In [ ]:
plot_output_ranges(quantile_outputs,targets, ['cdr'], quantiles, 1, 2000, 2025)

In [ ]:
import estival.priors as esp

In [ ]:
def convert_to_numpyro(prior):
    """Converts a UniformPrior object to a Numpyro distribution using its bounds."""
    low, high = prior.bounds()  # Using the bounds() method directly
    return dist.Uniform(low, high)

In [ ]:
def convert_all_priors_to_numpyro(priors):
    numpyro_priors = {}
    for key, prior in priors.items():
        numpyro_priors[key] = convert_to_numpyro(prior)
    return numpyro_priors

In [ ]:
numpyro_priors = convert_all_priors_to_numpyro(bcm.priors)

# You can now print or use these numpyro distributions
for name, dist in numpyro_priors.items():
    print(f"{name}: {dist}")

In [ ]:
def plot_post_prior_comparison(
    idata: az.InferenceData, 
    req_vars: List[str], 
    priors: List[dist.Distribution],
):
    """Plot comparison of model posterior outputs against priors.

    Args:
        idata: Arviz inference data from calibration
        req_vars: User-requested variables to plot
        priors: List of Numpyro prior distribution objects

    Returns:
        The figure object
    """
    # Determine the number of rows needed for two columns
    num_vars = len(req_vars)
    num_rows = (num_vars + 1) // 2  # This ensures an even distribution across two columns

    # Create the density plot with specified grid dimensions
    plot = az.plot_density(
        idata, 
        var_names=req_vars, 
        shade=0.3, 
        grid=(num_rows, 2)  # Set the grid to have num_rows rows and 2 columns
    )

    # Overlay the prior distributions
    for i_ax, ax in enumerate(plot.ravel()):
        if i_ax < len(req_vars):  # Ensure we don't exceed the number of requested variables
            ax_limits = ax.get_xlim()
            x_vals = np.linspace(*ax_limits, 50)
            y_vals = np.exp(priors[i_ax].log_prob(x_vals))
            y_vals *= ax.get_ylim()[1] / max(y_vals)  # Normalize prior to the plot's y-axis
            ax.fill_between(x_vals, y_vals, color="k", alpha=0.2, linewidth=2)

    plt.show()

In [ ]:
plot_post_prior_comparison(idata, list(numpyro_priors.keys()), [numpyro_priors[var] for var in list(numpyro_priors.keys())])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Define the range
a, b = 1.0, 5.0
mean = (a + b) / 2
stdev = 1e6  # large stdev for the truncated normal

# Uniform distribution
uniform_dist = stats.uniform(a, b - a)

# Truncated normal distribution
a_norm, b_norm = (a - mean) / stdev, (b - mean) / stdev
trunc_normal_dist = stats.truncnorm(a_norm, b_norm, loc=mean, scale=stdev)

# Generate x values
x = np.linspace(a, b, 1000)

# Compute PDFs
pdf_uniform = uniform_dist.pdf(x)
pdf_trunc_normal = trunc_normal_dist.pdf(x)

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(x, pdf_uniform, label='Uniform Distribution', linestyle='--')
plt.plot(x, pdf_trunc_normal, label='Truncated Normal Distribution', alpha=0.7)
plt.fill_between(x, pdf_trunc_normal, color='blue', alpha=0.1)
plt.fill_between(x, pdf_uniform, color='green', alpha=0.1)
plt.title('Comparison of Uniform vs. Very Large Stdev Truncated Normal Distribution')
plt.xlabel('Value')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True)
plt.show()